# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZainUlAbideen02/flyrank-ml-internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

Task Type: Binary Classification & Ranking

Framing: We frame Content Refresh Prioritization as a supervised binary classification problem (predicting whether a page will decay or not) combined with a Learning-to-Rank (LTR) approach to order candidate pages by decay probability for human editorial review.

In [9]:
import os, subprocess, sys, pandas as pd

# Clone starter repository data if missing from Colab session
if not os.path.exists("data/raw/content_refresh_anonymized.csv"):
    REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
    REPO_DIR = "flyrank-ml-internship-starter"

    if not os.path.isdir(REPO_DIR):
        print("Cloning repository dataset...")
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)

    os.chdir(REPO_DIR)

data_path = "data/raw/content_refresh_anonymized.csv"
print(f"Working Directory: {os.getcwd()}")
print(f"Dataset path verified: {os.path.exists(data_path)}")

Cloning repository dataset...
Working Directory: /content/flyrank-ml-internship-starter
Dataset path verified: True


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

Target Definition: is_declining (Binary Target: 1 if trend_direction == 'down', else 0).

Proxy Rationale: Since organic traffic decay is continuous, we use historical search performance trends (trend_pct < 0 and position degradation) as a proxy label for content freshness decay.

In [10]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

# Create the binary target column
df["is_declining"] = (df["trend_direction"] == "down").astype(int)

print("Target Column ('is_declining') Class Distribution:")
print(df["is_declining"].value_counts(normalize=True).round(3))

Target Column ('is_declining') Class Distribution:
is_declining
1    0.542
0    0.458
Name: proportion, dtype: float64


## 3. Success metric

*One metric you can defend. What number means 'good'?*

Primary Metric: Precision@K (specifically Precision@50 and Precision@100).Business Rationale: Content teams have limited time to review pages. Precision@K measures what percentage of the top $K$ pages recommended by our model actually needed a refresh. This directly optimizes human editor efficiency.

In [11]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Baseline Precision@50 dummy check logic demonstration
top_50_sample = df.head(50)
baseline_p50 = top_50_sample["is_declining"].mean()
print(f"Sample Top-50 Baseline Precision: {baseline_p50:.3f}")

Sample Top-50 Baseline Precision: 0.680


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

Unit of Analysis: One row = One unique URL / Content Page (content_id) for a specific client (client_id).

Dataframe Schema Preview: Each row aggregates historical search performance metrics (impressions, clicks, average position, CTR, word count, age) over a 90-day window.

In [12]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Show unit of analysis preview dataframe
unit_df = df[["content_id", "client_id", "search_volume", "avg_position", "ctr", "word_count", "is_declining"]].head(5)
print(f"Dataframe Shape: {df.shape[0]} rows (pages) x {df.shape[1]} columns")
unit_df

Dataframe Shape: 30000 rows (pages) x 45 columns


,content_id,client_id,search_volume,avg_position,ctr,word_count,is_declining
0,content_304f48230142,client_f369cb89fc,10.0,10.6,0.76,3221.0,1
1,content_a1fb4e703a9e,client_4e07408562,90.0,20.3,0.05,2481.0,1
2,content_9aa793d4d895,client_7f2253d7e2,0.0,36.5,0.09,3515.0,1
3,content_331d6c4de07b,client_19581e27de,10.0,6.2,0.49,NaN,0
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,44.0,0.13,2803.0,1


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

Why ML Beats Fixed Rules:

A simple hand-written rule (e.g., "refresh any page where avg_position > 20 and word_count < 1000") is static and fails to catch complex, non-linear interactions across multiple variables like CTR collapse, search volume changes, and category-level trends. Machine learning models (like Random Forests or Gradient Boosted Trees) adaptively weigh these feature interactions and generalize better across diverse client portfolios without manual rule tweaking.

In [13]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Feature interaction summary example
df.groupby("is_declining")[["avg_position", "ctr", "word_count"]].mean().round(2)

,avg_position,ctr,word_count
is_declining,,,
0,16.82,0.73,2957.45
1,15.94,0.32,3221.83


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.